# CoatShield boundary finder: train on Google Colab

Same training as the Kaggle notebook: pretrain the compact U-Net on OCT5k, fine-tune on
synthetic pellet scans, train the same network from scratch for comparison, evaluate,
export ONNX (opset 17) and write the manifest.

Before running:
1. On the laptop run `python scripts/package_for_kaggle.py`. It writes three zips to `dist/`.
2. Upload the three zips to a folder named `coatshield` in your Google Drive
   (`coatshield_code.zip`, `coatshield_synthetic.zip`, `oct5k.zip`). Keep the folder private.
3. Runtime > Change runtime type > T4 GPU, then Runtime > Run all. Allow the Drive access prompt.

The result is saved back to the same Drive folder as `coatshield_models.zip`.
The locked test set is never uploaded.

In [ ]:
import os, subprocess, zipfile
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/coatshield"   # folder holding the three zips
WORK = "/content/coatshield"

for name in ("coatshield_code.zip", "coatshield_synthetic.zip", "oct5k.zip"):
    assert os.path.exists(f"{DRIVE}/{name}"), f"{name} is not in {DRIVE}"
if not os.path.exists(WORK):
    zipfile.ZipFile(f"{DRIVE}/coatshield_code.zip").extractall("/content")
os.chdir(WORK)
if not os.path.exists("data/synthetic/train/images.npy"):
    zipfile.ZipFile(f"{DRIVE}/coatshield_synthetic.zip").extractall("data")        # data/synthetic/...
if not os.path.exists("data/raw/oct5k/OCT5k/Images"):
    zipfile.ZipFile(f"{DRIVE}/oct5k.zip").extractall("data/raw/oct5k")             # .../OCT5k/...
# torch and torchvision are preinstalled on Colab; install the rest.
subprocess.run("grep -v -E '^(torch|torchvision|--extra-index-url)' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt", shell=True, check=True)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - set the runtime type to T4 GPU")

## 1. Pretrain on OCT5k (split by patient volume)

In [ ]:
import json
from coatshield.config import load_config
from coatshield.seg.train import run_stage

cfg = load_config()
device = "cuda" if torch.cuda.is_available() else "cpu"
pre = run_stage("pretrain", cfg, device, workers=2)
print(json.dumps({k: v for k, v in pre.items() if k != "history"}, indent=1))

## 2. Fine-tune on synthetic pellets, and 3. the same network from scratch

In [ ]:
fine = run_stage("finetune", cfg, device, workers=2)
scratch = run_stage("scratch", cfg, device, workers=2)
for name, info in (("fine-tuned", fine), ("from scratch", scratch)):
    print(name, info["val_boundaries"], "parameters", info["parameters"])

## 4. Evaluate, export ONNX and write the manifest

In [ ]:
subprocess.run(["python", "scripts/evaluate_seg.py"], check=True)
subprocess.run(["python", "scripts/export_onnx.py", "--checkpoint", "unet_pellets", "--name", "unet"], check=True)
print(open("models/manifest.json").read())

## 5. Save the outputs to Drive

Download `coatshield_models.zip` from the Drive folder and unpack it at the repository root: it holds `models/unet.onnx`, `models/manifest.json`, the checkpoints and the evaluation report.

In [ ]:
subprocess.run(f"zip -q -r {DRIVE}/coatshield_models.zip models reports/seg_*", shell=True, check=True)
print(os.path.getsize(f"{DRIVE}/coatshield_models.zip") / 1e6, "MB saved to", DRIVE)